# XGBoost: four Ddc/Yb input ablations

Uses the same normalized datasets as the U-Net and transformer packages. Select case 30, 39, or 57. One scalar booster per target entry, with recoverable UBJ checkpoints.


In [1]:
from pathlib import Path
import subprocess
import sys
import json

CODE_DIR = Path.cwd()  # This extracted code folder
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 57
TARGET = 'dq'  # 'dq' or 'vmag'
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
DEVICE = 'cuda'  # or 'cuda:0'; use 'cpu' for local checks
ROUNDS = 600
MAX_DEPTH = 6
LEARNING_RATE = 0.05
EARLY_STOPPING = 20
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 1.0
THREADS = 4
SMOKE = False   # True tests tiny subsets and only 12 output entries
RESUME = False

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'gpu_xgboost_vmag.py').is_file(), 'Set CODE_DIR to the updated code folder'
for split in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{split}.npz').is_file()
outputs = CASE**2 if TARGET == 'dq' else CASE
print('Full run: up to', outputs * len(VARIANTS) * len(SEEDS), 'scalar models (constant outputs are skipped).')
print('Data:', DATA_DIR, '\nResults:', OUT_DIR)


Full run: up to 38988 scalar models (constant outputs are skipped).
Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57 
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/results/case57_dq


In [2]:
command = [sys.executable, '-u', str(CODE_DIR / 'gpu_xgboost_vmag.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--variants', *VARIANTS, '--target', TARGET, '--case', str(CASE),
    '--seeds', *map(str, SEEDS), '--device', DEVICE, '--threads', str(THREADS),
    '--rounds', str(ROUNDS), '--max-depth', str(MAX_DEPTH),
    '--learning-rate', str(LEARNING_RATE), '--early-stopping', str(EARLY_STOPPING),
    '--subsample', str(SUBSAMPLE), '--colsample-bytree', str(COLSAMPLE_BYTREE)]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/gpu_xgboost_vmag.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/results/case57_dq --variants ddc ddc_yb ddc_imag ddc_real --target dq --case 57 --seeds 42 123 2026 --device cuda --threads 4 --rounds 600 --max-depth 6 --learning-rate 0.05 --early-stopping 20 --subsample 0.8 --colsample-bytree 1.0
XGBoost 3.2.0 device cuda case 57 target dq

ddc seed=42: 3249 features, 3249 outputs; 0 completed
  completed 10/3249 outputs
  completed 20/3249 outputs
  completed 30/3249 outputs
  completed 40/3249 outputs
[heartbeat] XGBoost training is active
  completed 50/3249 outputs
  completed 60/3249 outputs
  completed 70/3249 outputs
  completed 80/3249 outputs
[heartbeat] XGBoost training is active
  completed 90/3249 outputs
  completed 100/3249 outputs
  co

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/gpu_xgboost_vmag.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/results/case57_dq', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--case', '57', '--seeds', '42', '123', '2026', '--device', 'cuda', '--threads', '4', '--rounds', '600', '--max-depth', '6', '--learning-rate', '0.05', '--early-stopping', '20', '--subsample', '0.8', '--colsample-bytree', '1.0'], returncode=0)

In [3]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Full target benchmark:', summary['protocol']['complete_target'])
for variant, splits in summary['aggregate'].items():
    print(variant, 'test:', splits['test'])
print('Raw and standardized metrics:', OUT_DIR / 'comparison.csv')


Full target benchmark: True
ddc test: {'r2': {'mean': 0.8368688571141142, 'std_across_seeds': 2.786278710794617e-05}, 'rmse': {'mean': 2.567147872877402, 'std_across_seeds': 0.0002192304393086106}, 'mae': {'mean': 0.9872994299817579, 'std_across_seeds': 0.00016506367140130946}, 'rmse_standardized': {'mean': 0.40556461374623987, 'std_across_seeds': 3.463458789672994e-05}, 'mae_standardized': {'mean': 0.15597610490212463, 'std_across_seeds': 2.6077195844133955e-05}}
ddc_yb test: {'r2': {'mean': 0.8382742025085537, 'std_across_seeds': 2.9741741914927235e-05}, 'rmse': {'mean': 2.5560661954810517, 'std_across_seeds': 0.00023503053613032018}, 'mae': {'mean': 0.9858062387675407, 'std_across_seeds': 0.0003127286848092703}, 'rmse_standardized': {'mean': 0.4038139019069946, 'std_across_seeds': 3.7130723795073144e-05}, 'mae_standardized': {'mean': 0.15574020670359182, 'std_across_seeds': 4.940568511354834e-05}}
ddc_imag test: {'r2': {'mean': 0.8378395185028754, 'std_across_seeds': 9.5834239954739

## Interpretation

Raw-unit R2/RMSE/MAE are comparable with the updated neural-network packages on the same case/split/target. Standardized metrics are separately labelled. Smoke mode evaluates a subset of output entries and is not a full-matrix accuracy result. Test data does not choose the model rounds or seed. See README.md for inference and resume details.
